# S6E9 · V9：RealMLP 10-fold OOF + 与 V8 相关性分析 + Nested Hill

这版只新训练 **RealMLP**，不会重训 CatBoost / LightGBM / old XGB / V8 strong XGB。

流程：

1. 读取你已经保存的 V3/V4/V5/V8 predictions；
2. 用官方 `pytabkit.RealMLP_TD_Classifier` 跑 10-fold OOF；
3. 重建 V8 nested-hill reference；
4. 计算 `RealMLP vs V8` 的 Spearman / Pearson；
5. 把 RealMLP 加进 engine garage；
6. 再跑一次 nested Caruana hill climb；
7. 自动决定是否用 V9 替换 V8。

公开的 S6E9 RealMLP notebook曾报告 Public 约 `0.94621`。这里使用官方 RealMLP-TD，而不是普通手写 MLP。

## 1. 环境

第一次运行如果缺少 `pytabkit`，会自动安装。

- CUDA 可用：自动用 CUDA
- Mac：默认 CPU，不强行使用 MPS

In [4]:
from pathlib import Path
from datetime import datetime
import importlib.util, subprocess, sys, json, gc

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from scipy.stats import rankdata, norm

if importlib.util.find_spec("pytabkit") is None:
    print("Installing pytabkit...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "pytabkit"])

try:
    from pytabkit import RealMLP_TD_Classifier
except ImportError:
    from pytabkit.models.sklearn.sklearn_interfaces import RealMLP_TD_Classifier

import torch

print("Python:", sys.executable)
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

Python: /opt/miniconda3/envs/dl/bin/python
Torch: 2.14.0
CUDA: False


## 2. 配置与路径

V8 strong XGB 目录默认自动寻找最新的完整目录。

In [5]:
PROJECT = Path("/Users/yw/unsw/Kaggle")
TARGET = "Will_Buy_EV"

CAT_DIR = PROJECT / "outputs/v3_20260925_180457_278867"
LGB_DIR = PROJECT / "outputs/v4_lightgbm_20260930_163526_079778"
OLD_XGB_DIR = PROJECT / "outputs/v5_xgb10_20260930_205349_440980"

V8_STRONG_DIR = None
REALMLP_RESULT_DIR = None

REALMLP_SPLITS = 10
REALMLP_SEED = 42
REALMLP_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
REALMLP_EPOCHS = 256
REALMLP_BATCH_SIZE = 256
REALMLP_HIDDEN = [256, 256, 256]
REALMLP_LR = 0.04

META_FOLDS = 5
META_SEED = 2026
HILL_MAX_STEPS = 80
HILL_TOL = 1e-7

print("PROJECT:", PROJECT)
print("RealMLP device:", REALMLP_DEVICE)

PROJECT: /Users/yw/unsw/Kaggle
RealMLP device: cpu


## 3. 读取数据和旧 prediction

In [6]:
def require(condition, message):
    if not bool(condition):
        raise ValueError(message)

def load_data(project):
    train = pd.read_csv(Path(project) / "data/train.csv")
    test = pd.read_csv(Path(project) / "data/test.csv")
    require(TARGET in train.columns, "Missing target")
    require(train["id"].is_unique and test["id"].is_unique, "Duplicate id")
    y = train[TARGET].map({"No": 0, "Yes": 1})
    require(y.notna().all(), "Expected No/Yes target")
    return train, test, y.astype(np.int8)

def align_prediction_csv(path, ids, pred_col):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    frame = pd.read_csv(path)
    require("id" in frame.columns and pred_col in frame.columns, f"Bad file: {path}")
    require(frame["id"].is_unique, f"Duplicate ids: {path}")
    wanted = pd.Index(np.asarray(ids))
    require(len(frame) == len(wanted), f"Wrong row count: {path}")
    require(wanted.isin(frame["id"]).all(), f"ID mismatch: {path}")
    out = frame.set_index("id").loc[wanted, pred_col].to_numpy(dtype=np.float64)
    require(np.isfinite(out).all(), f"Non-finite predictions: {path}")
    return out

def read_saved(run_dir, model_name, train, test):
    return {
        "oof": align_prediction_csv(
            Path(run_dir) / f"oof_{model_name}.csv",
            train["id"],
            "prediction",
        ),
        "test": align_prediction_csv(
            Path(run_dir) / f"submission_{model_name}.csv",
            test["id"],
            TARGET,
        ),
    }

def find_complete_v8_dir(project):
    root = Path(project) / "outputs"
    dirs = sorted(
        root.glob("v8_strong_xgb_*"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    needed = [
        "oof_strong_rich_te.csv",
        "submission_strong_rich_te.csv",
        "oof_strong_digit_freq.csv",
        "submission_strong_digit_freq.csv",
    ]
    for d in dirs:
        if all((d / f).is_file() for f in needed):
            return d
    raise FileNotFoundError("No complete v8_strong_xgb_* directory found")

def rank01(values):
    values = np.asarray(values, dtype=np.float64)
    return rankdata(values, method="average") / len(values)

def probit_rank(values):
    values = np.asarray(values, dtype=np.float64)
    r = rankdata(values, method="average")
    return norm.ppf(r / (len(values) + 1.0))

def save_submission(project, test, values, path):
    values = rank01(values)
    sub = pd.DataFrame({"id": test["id"].to_numpy(), TARGET: values})
    sample = Path(project) / "data/sample_submission.csv"
    if sample.is_file():
        ids = pd.read_csv(sample, usecols=["id"])
        sub = ids.merge(sub, on="id", how="left", validate="one_to_one", sort=False)
    require(sub[TARGET].notna().all(), "Missing submission predictions")
    sub.to_csv(path, index=False)

train, test, y = load_data(PROJECT)

if V8_STRONG_DIR is None:
    V8_STRONG_DIR = find_complete_v8_dir(PROJECT)
else:
    V8_STRONG_DIR = Path(V8_STRONG_DIR)

cat = read_saved(CAT_DIR, "catboost", train, test)
lgb = read_saved(LGB_DIR, "lightgbm", train, test)
old_xgb = read_saved(OLD_XGB_DIR, "xgboost", train, test)

strong_rich = {
    "oof": align_prediction_csv(V8_STRONG_DIR / "oof_strong_rich_te.csv", train["id"], "prediction"),
    "test": align_prediction_csv(V8_STRONG_DIR / "submission_strong_rich_te.csv", test["id"], TARGET),
}
strong_digit = {
    "oof": align_prediction_csv(V8_STRONG_DIR / "oof_strong_digit_freq.csv", train["id"], "prediction"),
    "test": align_prediction_csv(V8_STRONG_DIR / "submission_strong_digit_freq.csv", test["id"], TARGET),
}

check = pd.DataFrame([
    ["catboost", roc_auc_score(y, cat["oof"])],
    ["lightgbm", roc_auc_score(y, lgb["oof"])],
    ["old_xgb", roc_auc_score(y, old_xgb["oof"])],
    ["strong_rich_te", roc_auc_score(y, strong_rich["oof"])],
    ["strong_digit_freq", roc_auc_score(y, strong_digit["oof"])],
], columns=["engine", "oof_auc"]).sort_values("oof_auc", ascending=False)

print("V8 strong dir:", V8_STRONG_DIR)
print(check.to_string(index=False, float_format=lambda v: f"{v:.9f}"))

V8 strong dir: /Users/yw/unsw/Kaggle/outputs/v8_strong_xgb_20260930_213758_937864
           engine     oof_auc
   strong_rich_te 0.945998429
         lightgbm 0.945946381
          old_xgb 0.945931543
         catboost 0.945618233
strong_digit_freq 0.944402544


## 4. RealMLP 特征

为了增加与 GBDT 的差异，RealMLP 只用原始字段，不加 XGB 的 target encoding。

In [7]:
def make_realmlp_features(frame):
    x = frame.drop(columns=["id", TARGET], errors="ignore").copy()
    cat_cols = []
    for col in x.columns:
        if pd.api.types.is_object_dtype(x[col]) or pd.api.types.is_string_dtype(x[col]):
            x[col] = x[col].astype(str)
            cat_cols.append(col)
        else:
            x[col] = pd.to_numeric(x[col], errors="raise")
    require(not x.isna().any().any(), "RealMLP input has missing values")
    return x, cat_cols

X_real, REALMLP_CAT_COLS = make_realmlp_features(train)
X_real_test, REALMLP_CAT_COLS_TEST = make_realmlp_features(test)

require(X_real.columns.tolist() == X_real_test.columns.tolist(), "Feature columns differ")
require(REALMLP_CAT_COLS == REALMLP_CAT_COLS_TEST, "Categorical columns differ")

print("Features:", X_real.shape[1])
print("Categorical:", REALMLP_CAT_COLS)

Features: 13
Categorical: ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']


## 5. RealMLP 10-fold OOF

如果以后重新打开 notebook，不想再训练，把：

```python
REALMLP_RESULT_DIR = PROJECT / "outputs/v9_realmlp10_你的目录"
```

填到配置里即可复用。

In [8]:
def run_realmlp_10fold(project, X, X_test, y, cat_cols, result_dir=None):
    folds = list(
        StratifiedKFold(
            n_splits=REALMLP_SPLITS,
            shuffle=True,
            random_state=REALMLP_SEED,
        ).split(X, y)
    )

    if result_dir is not None:
        out = Path(result_dir)
        oof = align_prediction_csv(out / "oof_realmlp.csv", train["id"], "prediction")
        test_pred = align_prediction_csv(out / "submission_realmlp.csv", test["id"], TARGET)
        auc = float(roc_auc_score(y, oof))
        print("Reusing RealMLP:", out)
        print("RealMLP OOF AUC:", auc)
        return {"dir": out, "oof": oof, "test": test_pred, "auc": auc, "folds": folds}

    out = Path(project) / "outputs" / ("v9_realmlp10_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
    out.mkdir(parents=True, exist_ok=False)

    oof = np.full(len(X), np.nan, dtype=np.float64)
    coverage = np.zeros(len(X), dtype=np.int8)
    test_sum = np.zeros(len(X_test), dtype=np.float64)
    fold_rows = []

    for fold, (ti, vi) in enumerate(folds, start=1):
        print(f"\n========== RealMLP | Fold {fold}/{REALMLP_SPLITS} ==========", flush=True)

        model = RealMLP_TD_Classifier(
            device=REALMLP_DEVICE,
            random_state=REALMLP_SEED + fold,
            n_cv=1,
            n_refit=0,
            n_epochs=REALMLP_EPOCHS,
            batch_size=REALMLP_BATCH_SIZE,
            hidden_sizes=REALMLP_HIDDEN,
            val_metric_name="1-auc_ovr",
            use_ls=False,
            lr=REALMLP_LR,
            verbosity=1,
        )

        X_tr = X.iloc[ti].copy()
        X_va = X.iloc[vi].copy()
        y_tr = np.asarray(y)[ti]
        y_va = np.asarray(y)[vi]

        model.fit(
            X_tr,
            y_tr,
            X_va,
            y_va,
            cat_col_names=cat_cols,
        )

        vp = np.asarray(model.predict_proba(X_va))[:, 1]
        tp = np.asarray(model.predict_proba(X_test))[:, 1]

        oof[vi] = vp
        coverage[vi] += 1
        test_sum += tp

        fold_auc = float(roc_auc_score(y_va, vp))
        fold_rows.append({"fold": fold, "auc": fold_auc})
        pd.DataFrame(fold_rows).to_csv(out / "fold_scores_realmlp.csv", index=False)

        np.savez_compressed(
            out / f"realmlp_fold{fold}_predictions.npz",
            valid_indices=vi,
            valid_ids=train["id"].iloc[vi].to_numpy(),
            valid_prediction=vp,
            test_ids=test["id"].to_numpy(),
            test_prediction=tp,
        )

        print(f"Fold {fold} AUC: {fold_auc:.9f}")

        del model, X_tr, X_va, y_tr, y_va, vp, tp
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    require((coverage == 1).all(), "OOF coverage error")
    require(np.isfinite(oof).all(), "OOF contains NaN")

    test_pred = test_sum / REALMLP_SPLITS
    total_auc = float(roc_auc_score(y, oof))

    pd.DataFrame({
        "id": train["id"].to_numpy(),
        "target": np.asarray(y),
        "prediction": oof,
    }).to_csv(out / "oof_realmlp.csv", index=False)

    pd.DataFrame({
        "id": test["id"].to_numpy(),
        TARGET: test_pred,
    }).to_csv(out / "submission_realmlp.csv", index=False)

    (out / "config.json").write_text(
        json.dumps({
            "model": "RealMLP_TD_Classifier",
            "folds": REALMLP_SPLITS,
            "seed": REALMLP_SEED,
            "device": REALMLP_DEVICE,
            "epochs": REALMLP_EPOCHS,
            "batch_size": REALMLP_BATCH_SIZE,
            "hidden_sizes": REALMLP_HIDDEN,
            "lr": REALMLP_LR,
            "oof_auc": total_auc,
        }, indent=2),
        encoding="utf-8",
    )

    print("\nRealMLP 10-fold OOF AUC:", total_auc)
    print("V9 RealMLP directory:", out)

    return {"dir": out, "oof": oof, "test": test_pred, "auc": total_auc, "folds": folds}

if REALMLP_RESULT_DIR is None:
    realmlp = run_realmlp_10fold(PROJECT, X_real, X_real_test, y, REALMLP_CAT_COLS)
    REALMLP_RESULT_DIR = realmlp["dir"]
else:
    realmlp = run_realmlp_10fold(
        PROJECT, X_real, X_real_test, y, REALMLP_CAT_COLS,
        result_dir=REALMLP_RESULT_DIR,
    )


========== RealMLP | Fold 1/10 ==========
Columns classified as continuous: ['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level']
Columns classified as categorical: ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']


GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/opt/miniconda3/envs/dl/lib/python3.11/site-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/opt/miniconda3/envs/dl/lib/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
`Trainer.fit` stopped: `max_epochs=256` reached.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/opt/miniconda3/envs/dl/lib/python3.11/site-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`

Fold 1 AUC: 0.939000969

========== RealMLP | Fold 2/10 ==========
Columns classified as continuous: ['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level']
Columns classified as categorical: ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']


GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/opt/miniconda3/envs/dl/lib/python3.11/site-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/opt/miniconda3/envs/dl/lib/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
`Trainer.fit` stopped: `max_epochs=256` reached.
GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/opt/miniconda3/envs/dl/lib/python3.11/site-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`

Fold 2 AUC: 0.938924713

========== RealMLP | Fold 3/10 ==========
Columns classified as continuous: ['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level']
Columns classified as categorical: ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible', 'Subsidy_Available', 'Range_Anxiety_Level']


GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/opt/miniconda3/envs/dl/lib/python3.11/site-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/opt/miniconda3/envs/dl/lib/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

/opt/miniconda3/envs/dl/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3831: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## 6. 重建 V8 reference + Nested Hill

In [ ]:
def build_v8_engines():
    current_oof = 0.5 * cat["oof"] + 0.5 * lgb["oof"]
    current_test = 0.5 * cat["test"] + 0.5 * lgb["test"]

    v5_oof = 0.5 * rank01(old_xgb["oof"]) + 0.5 * rank01(current_oof)
    v5_test = 0.5 * rank01(old_xgb["test"]) + 0.5 * rank01(current_test)

    eo = {
        "catboost": cat["oof"],
        "lightgbm": lgb["oof"],
        "old_xgboost": old_xgb["oof"],
        "current_cat_lgb": current_oof,
        "v5_baseline": v5_oof,
        "strong_rich_te": strong_rich["oof"],
        "strong_rich_te__current_rank50": 0.5 * rank01(strong_rich["oof"]) + 0.5 * rank01(current_oof),
        "strong_digit_freq": strong_digit["oof"],
        "strong_digit_freq__current_rank50": 0.5 * rank01(strong_digit["oof"]) + 0.5 * rank01(current_oof),
        "v8_views_equal_rank": 0.5 * rank01(strong_rich["oof"]) + 0.5 * rank01(strong_digit["oof"]),
    }
    et = {
        "catboost": cat["test"],
        "lightgbm": lgb["test"],
        "old_xgboost": old_xgb["test"],
        "current_cat_lgb": current_test,
        "v5_baseline": v5_test,
        "strong_rich_te": strong_rich["test"],
        "strong_rich_te__current_rank50": 0.5 * rank01(strong_rich["test"]) + 0.5 * rank01(current_test),
        "strong_digit_freq": strong_digit["test"],
        "strong_digit_freq__current_rank50": 0.5 * rank01(strong_digit["test"]) + 0.5 * rank01(current_test),
        "v8_views_equal_rank": 0.5 * rank01(strong_rich["test"]) + 0.5 * rank01(strong_digit["test"]),
    }
    return eo, et

def matrices(eo, et):
    names = list(eo)
    require(names == list(et), "Engine names differ")
    P_oof = np.column_stack([probit_rank(eo[n]) for n in names])
    P_test = np.column_stack([probit_rank(et[n]) for n in names])
    return names, P_oof, P_test

def hill_select(P, labels):
    labels = np.asarray(labels, dtype=np.int8)
    blend = np.zeros(len(labels), dtype=np.float64)
    counts = np.zeros(P.shape[1], dtype=np.float64)
    best = -np.inf

    for _ in range(HILL_MAX_STEPS):
        scores = [roc_auc_score(labels, blend + P[:, j]) for j in range(P.shape[1])]
        j = int(np.argmax(scores))
        score = float(scores[j])
        if score <= best + HILL_TOL:
            break
        blend += P[:, j]
        counts[j] += 1
        best = score

    require(counts.sum() > 0, "Hill selected nothing")
    return counts / counts.sum()

def run_nested_hill(P_oof, P_test, names, y, reference_oof=None):
    folds = list(
        StratifiedKFold(META_FOLDS, shuffle=True, random_state=META_SEED).split(P_oof, y)
    )
    oof = np.zeros(len(y), dtype=np.float64)
    rows = []

    for fold, (tr, va) in enumerate(folds, start=1):
        w = hill_select(P_oof[tr], np.asarray(y)[tr])
        pred = P_oof[va] @ w
        oof[va] = pred
        row = {
            "fold": fold,
            "auc": float(roc_auc_score(np.asarray(y)[va], pred)),
            "engines_used": int((w > 0).sum()),
        }
        if reference_oof is not None:
            ref_auc = float(roc_auc_score(np.asarray(y)[va], np.asarray(reference_oof)[va]))
            row["reference_auc"] = ref_auc
            row["delta_vs_reference"] = row["auc"] - ref_auc
        rows.append(row)

    w_full = hill_select(P_oof, y)
    return {
        "oof": oof,
        "test": P_test @ w_full,
        "auc": float(roc_auc_score(y, oof)),
        "fold_table": pd.DataFrame(rows),
        "weights": pd.Series(w_full, index=names),
    }

v8_eo, v8_et = build_v8_engines()
v8_names, v8_P_oof, v8_P_test = matrices(v8_eo, v8_et)
v8_ref = run_nested_hill(v8_P_oof, v8_P_test, v8_names, y)

print("Reconstructed V8 nested-hill OOF:", v8_ref["auc"])
print(v8_ref["weights"].loc[lambda s: s > 0].sort_values(ascending=False).to_string())

## 7. RealMLP 与 V8 相关性

In [ ]:
def corr(a, b, method):
    return float(pd.Series(a).corr(pd.Series(b), method=method))

corr_table = pd.DataFrame([
    {
        "pair": "RealMLP vs V8_nested_hill",
        "spearman": corr(realmlp["oof"], v8_ref["oof"], "spearman"),
        "pearson": float(np.corrcoef(realmlp["oof"], v8_ref["oof"])[0, 1]),
    },
    {
        "pair": "RealMLP vs strong_rich_te",
        "spearman": corr(realmlp["oof"], strong_rich["oof"], "spearman"),
        "pearson": float(np.corrcoef(realmlp["oof"], strong_rich["oof"])[0, 1]),
    },
    {
        "pair": "RealMLP vs old_XGB",
        "spearman": corr(realmlp["oof"], old_xgb["oof"], "spearman"),
        "pearson": float(np.corrcoef(realmlp["oof"], old_xgb["oof"])[0, 1]),
    },
])

print("RealMLP OOF:", realmlp["auc"])
print("V8 reference OOF:", v8_ref["auc"])
print("\nCorrelations:")
print(corr_table.to_string(index=False, float_format=lambda v: f"{v:.9f}"))

## 8. 加入 RealMLP 后重新 Nested Hill

In [ ]:
v9_eo = dict(v8_eo)
v9_et = dict(v8_et)

v9_eo["realmlp"] = realmlp["oof"]
v9_et["realmlp"] = realmlp["test"]

v9_eo["realmlp__v8_rank50"] = 0.5 * rank01(realmlp["oof"]) + 0.5 * rank01(v8_ref["oof"])
v9_et["realmlp__v8_rank50"] = 0.5 * rank01(realmlp["test"]) + 0.5 * rank01(v8_ref["test"])

v9_names, v9_P_oof, v9_P_test = matrices(v9_eo, v9_et)

v9_nested = run_nested_hill(
    v9_P_oof,
    v9_P_test,
    v9_names,
    y,
    reference_oof=v8_ref["oof"],
)

folds = v9_nested["fold_table"]
wins = int((folds["delta_vs_reference"] > 0).sum())
mean_delta = float(folds["delta_vs_reference"].mean())
overall_delta = float(v9_nested["auc"] - v8_ref["auc"])

accepted = bool(overall_delta > 0 and wins >= 4 and mean_delta > 0)

print("V8 reference OOF:", v8_ref["auc"])
print("RealMLP OOF:", realmlp["auc"])
print("V9 nested hill OOF:", v9_nested["auc"])
print("Delta vs V8:", overall_delta)
print(f"Improved folds vs V8: {wins}/{META_FOLDS}")
print("Mean fold delta:", mean_delta)
print("V9 accepted:", accepted)

print("\nPer-fold:")
print(folds.to_string(index=False, float_format=lambda v: f"{v:.9f}"))

print("\nV9 weights:")
print(v9_nested["weights"].loc[lambda s: s > 0].sort_values(ascending=False).to_string())

## 9. 简单 V8 + RealMLP rank blend

只测试 10%、20%、30%、50% RealMLP，帮助判断 NN 是否真的提供补充信息。

In [ ]:
v8_ro = rank01(v8_ref["oof"])
v8_rt = rank01(v8_ref["test"])
rm_ro = rank01(realmlp["oof"])
rm_rt = rank01(realmlp["test"])

blend_rows = []
blend_preds = {}

for rw in (0.10, 0.20, 0.30, 0.50):
    vw = 1.0 - rw
    name = f"v8_{int(vw*100)}_realmlp_{int(rw*100)}"
    oof_pred = vw * v8_ro + rw * rm_ro
    test_pred = vw * v8_rt + rw * rm_rt
    auc = float(roc_auc_score(y, oof_pred))
    blend_rows.append({
        "candidate": name,
        "oof_auc": auc,
        "delta_vs_v8": auc - v8_ref["auc"],
    })
    blend_preds[name] = {"oof": oof_pred, "test": test_pred}

blend_table = pd.DataFrame(blend_rows).sort_values("oof_auc", ascending=False)
print(blend_table.to_string(index=False, float_format=lambda v: f"{v:.9f}"))

## 10. 保存 V9 submission

`submission_v9_next_candidate.csv` 只在 V9 nested hill 通过 4/5 fold gate 时选择 V9，否则回退到 V8。

In [ ]:
out = PROJECT / "outputs" / ("v9_final_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
out.mkdir(parents=True, exist_ok=False)

save_submission(PROJECT, test, realmlp["test"], out / "submission_v9_realmlp_single.csv")
save_submission(PROJECT, test, v8_ref["test"], out / "submission_v9_v8_reference.csv")
save_submission(PROJECT, test, v9_nested["test"], out / "submission_v9_nested_hill.csv")

for name, info in blend_preds.items():
    save_submission(PROJECT, test, info["test"], out / f"submission_v9_{name}.csv")

if accepted:
    selected_name = "v9_nested_hill"
    selected_test = v9_nested["test"]
    selected_auc = v9_nested["auc"]
else:
    selected_name = "v8_reference_fallback"
    selected_test = v8_ref["test"]
    selected_auc = v8_ref["auc"]

save_submission(PROJECT, test, selected_test, out / "submission_v9_next_candidate.csv")

rows = [
    {"candidate": "V8_reference", "oof_auc": v8_ref["auc"], "delta_vs_v8": 0.0},
    {"candidate": "RealMLP_single", "oof_auc": realmlp["auc"], "delta_vs_v8": realmlp["auc"] - v8_ref["auc"]},
    {"candidate": "V9_nested_hill", "oof_auc": v9_nested["auc"], "delta_vs_v8": v9_nested["auc"] - v8_ref["auc"]},
] + blend_rows

summary = pd.DataFrame(rows).sort_values("oof_auc", ascending=False).reset_index(drop=True)
summary.to_csv(out / "v9_comparison.csv", index=False)
corr_table.to_csv(out / "v9_correlations.csv", index=False)
folds.to_csv(out / "v9_nested_hill_folds.csv", index=False)

(out / "v9_report.json").write_text(
    json.dumps({
        "v8_reference_auc": float(v8_ref["auc"]),
        "realmlp_auc": float(realmlp["auc"]),
        "v9_nested_auc": float(v9_nested["auc"]),
        "v9_delta_vs_v8": overall_delta,
        "v9_wins_vs_v8": wins,
        "v9_mean_fold_delta": mean_delta,
        "v9_accepted": accepted,
        "selected": selected_name,
        "selected_oof_auc": float(selected_auc),
        "realmlp_result_dir": str(REALMLP_RESULT_DIR),
        "v8_strong_dir": str(V8_STRONG_DIR),
    }, indent=2),
    encoding="utf-8",
)

print("================ V9 FINAL ================")
print(summary.to_string(index=False, float_format=lambda v: f"{v:.9f}"))
print("\nCorrelations:")
print(corr_table.to_string(index=False, float_format=lambda v: f"{v:.9f}"))
print("\nV9 accepted:", accepted)
print("Selected:", selected_name)
print("Output:", out)
print("Next candidate:", out / "submission_v9_next_candidate.csv")

## 11. 跑完后把这些发给我

```text
RealMLP 10-fold OOF AUC
Reconstructed V8 nested-hill OOF
RealMLP vs V8_nested_hill Spearman

V9 nested hill OOF
Delta vs V8
Improved folds vs V8
V9 accepted

V9 FINAL
```

判断重点：

- RealMLP 单模不一定要第一；
- 如果 Spearman 明显低于树模型之间的相关性，并且 V9 nested hill 提升，就是成功；
- 如果 RealMLP 低、相关性也不够低、V9 不涨，下一步再换 TabM / Transformer / BlamerX-style feature view。